# Project 5 — MOJ Criminal Court Statistics · **Gold layer**

Silver gave me clean, tidy tables. **Gold is where I answer specific questions.** Each
table below is small, filtered to one geography level, and shaped for a single chart or
story — these are the cuts that become the article series and the Streamlit app.

Nothing new is invented: Gold is just Silver, filtered and summed. Above each cut I say
which question it answers, what I filter to, what I group by, and the columns I create.
Two rules I keep: (1) filter to ONE `geo_level` so I never double-count across levels,
and (2) detect the latest quarter dynamically, so this notebook still works next release.

## Step 0 — Setup and a tiny save helper

Gold tables are small, so I save each one as **both Parquet and CSV** (CSV is handy for
Medium charts and Streamlit). `save_gold(df, name)` does both.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

SILVER = PROJECT_DIR / "data" / "silver"
GOLD   = PROJECT_DIR / "data" / "gold"
GOLD.mkdir(parents=True, exist_ok=True)

def save_gold(df, name):
    df.to_parquet(GOLD / f"{name}.parquet", index=False)
    df.to_csv(GOLD / f"{name}.csv", index=False)
    print(f"saved {name}: {df.shape[0]} rows x {df.shape[1]} cols  (parquet + csv)")

print("Silver in:", SILVER)
print("Gold out :", GOLD)

## Step 1 — `gold_backlog_trajectory` (the road to 80,200)

**Question:** how have receipts, disposals and the open caseload moved each quarter,
nationally?

**How I build it:** from `silver_cc_rdos`, keep only `offence == "All offences"` (that's
the pre-computed all-offence total — I must not also add the individual offences), then
sum across every court and case type, grouped by quarter and measure. I pivot so each
quarter is one row with `receipts`, `disposals`, `open`, and add **`net_flow`** =
receipts − disposals (a positive net_flow means the backlog grows that quarter).

In [ ]:
rdos = pd.read_parquet(SILVER / "silver_cc_rdos.parquet",
        columns=["quarter_end", "year", "quarter", "offence", "measure", "count"])

nat = rdos[rdos.offence == "All offences"]
g = (nat.groupby(["quarter_end", "year", "quarter", "measure"], observed=True)["count"]
        .sum().reset_index())

traj = g.pivot_table(index=["quarter_end", "year", "quarter"],
                     columns="measure", values="count").reset_index()
traj.columns.name = None
traj = traj.rename(columns={"Receipts": "receipts", "Disposals": "disposals", "Open": "open"})
traj["net_flow"] = traj["receipts"] - traj["disposals"]   # >0 means backlog grew
traj = traj.sort_values("quarter_end").reset_index(drop=True)

save_gold(traj, "gold_backlog_trajectory")
traj.tail(4)

## Step 2 — `gold_caseload_age` + summary (the ageing caseload)

**Question:** how is the open caseload distributed by age, and how much of it is now a
year or more old?

**How I build it:** from `silver_cc_open_age`, filter to national, `All open cases`,
`All offences`, and the real age bands (`row_type == "age_band"`). That gives a tidy
quarter × age_band series. Then a small **summary** table per quarter with `total_cases`
(the Total-cases row), `valid_cases` (sum of bands), `open_1yr_plus` (the 1–2y + 2y+
bands) and **`share_1yr_plus`** (of valid cases).

In [ ]:
oa = pd.read_parquet(SILVER / "silver_cc_open_age.parquet",
        columns=["quarter_end","year","quarter","geo_level","case_type",
                 "offence","age_band","age_order","row_type","count"])

base = oa[(oa.geo_level == "national") & (oa.case_type == "All open cases") &
          (oa.offence == "All offences")]

# (a) tidy age-band series
age = (base[base.row_type == "age_band"]
       .groupby(["quarter_end","year","quarter","age_band","age_order"], observed=True)["count"]
       .sum().reset_index()
       .sort_values(["quarter_end","age_order"]))
save_gold(age, "gold_caseload_age")

# (b) per-quarter summary
def q_summary(df):
    total = df.loc[df.age_band == "Total cases", "count"].sum()
    valid = df.loc[df.row_type == "age_band", "count"].sum()
    yr1   = df.loc[df.age_band.isin(["1 to 2 years","2 years or more"]), "count"].sum()
    return pd.Series({"total_cases": total, "valid_cases": valid,
                      "open_1yr_plus": yr1,
                      "share_1yr_plus": (yr1 / valid) if valid else float("nan")})

summ = (base.groupby(["quarter_end","year","quarter"], observed=True)
            .apply(q_summary, include_groups=False).reset_index()
            .sort_values("quarter_end"))
save_gold(summ, "gold_caseload_age_summary")
summ.tail(4)

## Step 3 — `gold_oldest_by_offence` (who is waiting longest)

**Question:** of the cases open a year or more, which offences do they involve — and is
it true that sexual offences and violence make up over half?

**How I build it:** from `silver_cc_open_age`, national, `All open cases`, age bands
`1 to 2 years` + `2 years or more`, and the **component** offence groups only (I drop
`All offences` and the `… - Adult/Child/All Rape` sub-splits so nothing is
double-counted). Group by quarter × offence → **`open_1yr_plus`**, then add
**`share_of_quarter`** (each offence's slice of that quarter's 1yr-plus total).

In [ ]:
comp = base = pd.read_parquet(SILVER / "silver_cc_open_age.parquet",
        columns=["quarter_end","year","quarter","geo_level","case_type",
                 "offence","age_band","count"])
old = comp[(comp.geo_level == "national") & (comp.case_type == "All open cases") &
           (comp.age_band.isin(["1 to 2 years","2 years or more"])) &
           (comp.offence != "All offences") & (~comp.offence.str.contains(" - ", na=False))]

by_off = (old.groupby(["quarter_end","year","quarter","offence"], observed=True)["count"]
             .sum().reset_index().rename(columns={"count":"open_1yr_plus"}))
totals = by_off.groupby("quarter_end")["open_1yr_plus"].transform("sum")
by_off["share_of_quarter"] = by_off["open_1yr_plus"] / totals
by_off = by_off.sort_values(["quarter_end","open_1yr_plus"], ascending=[True, False])

save_gold(by_off, "gold_oldest_by_offence")
# peek at the latest quarter, ranked
latest = by_off.quarter_end.max()
by_off[by_off.quarter_end == latest].head(6)

## Step 4 — `gold_timeliness_trend` (justice getting slower)

**Question:** how long, end to end, is it taking a case to complete — and how is that
moving?

**How I build it:** from `silver_cc_timeliness`, national (`England and Wales`),
`All cases closed`, `All offences`. I keep the headline **offence-to-completion** mean
and median (in days) plus two useful stages, one row per quarter.

In [ ]:
tl = pd.read_parquet(SILVER / "silver_cc_timeliness.parquet",
        columns=["quarter_end","year","quarter","geo_level","case_type","offence",
                 "offence_to_completion_mean","offence_to_completion_median",
                 "charge_to_completion_at_the_crown_court_mean",
                 "charge_to_completion_at_the_crown_court_median"])

tt = tl[(tl.geo_level == "national") & (tl.case_type == "All cases closed") &
        (tl.offence == "All offences")].copy()
tt = (tt[["quarter_end","year","quarter",
          "offence_to_completion_mean","offence_to_completion_median",
          "charge_to_completion_at_the_crown_court_mean",
          "charge_to_completion_at_the_crown_court_median"]]
      .sort_values("quarter_end").reset_index(drop=True))

save_gold(tt, "gold_timeliness_trend")
tt.tail(4)

## Step 5 — `gold_backlog_by_court` (the geography cut)

**Question:** which court centres carry the biggest open caseloads, and how has each
changed over the last year?

**How I build it:** `silver_cc_rdos` is the only table with court-level detail. I take
`Open`, `All offences`, sum across case types per court, for the **latest** quarter and
the **same quarter a year earlier**, then join to get `open_latest`, `open_year_ago`,
`change` and `pct_change`. This feeds the map/table in Streamlit.

In [ ]:
rc = pd.read_parquet(SILVER / "silver_cc_rdos.parquet",
        columns=["quarter_end","year","quarter","region","court","offence","measure","count"])
op = rc[(rc.measure == "Open") & (rc.offence == "All offences")]

latest_q  = op.quarter_end.max()
year_ago  = latest_q - pd.offsets.DateOffset(years=1)

def court_open(qend):
    return (op[op.quarter_end == qend]
            .groupby(["region","court"], observed=True)["count"].sum())

cur  = court_open(latest_q).rename("open_latest")
prev = court_open(year_ago).rename("open_year_ago")
court = pd.concat([cur, prev], axis=1).reset_index()
court["change"]     = court["open_latest"] - court["open_year_ago"]
court["pct_change"] = court["change"] / court["open_year_ago"]
court = court.sort_values("open_latest", ascending=False).reset_index(drop=True)

save_gold(court, "gold_backlog_by_court")
print(f"latest quarter: {latest_q.date()}  vs year ago: {year_ago.date()}")
court.head(6)

## Step 6 — `gold_remand_age` (custody vs bail — who is left to age)

**Question:** among open *trial* cases, do people on remand in custody or on bail wait
longest? (I expected custody to age most; the data says the opposite — custody time
limits push those cases up the list, so it's **bail** cases that age.)

**How I build it:** from `silver_cc_open_age`, national, `All offences`, `case_type ==
"Trials"`, and `remand_status` in Custody / Bail / Unknown, real age bands only. A tidy
quarter × remand × age_band series, plus a summary with `open_1yr_plus` and its share.

In [ ]:
oar = pd.read_parquet(SILVER / "silver_cc_open_age.parquet",
        columns=["quarter_end","year","quarter","geo_level","case_type","remand_status",
                 "offence","age_band","age_order","row_type","count"])
rem = oar[(oar.geo_level == "national") & (oar.offence == "All offences") &
          (oar.case_type == "Trials") &
          (oar.remand_status.isin(["Custody","Bail","Unknown"])) &
          (oar.row_type == "age_band")]

remand_age = (rem.groupby(["quarter_end","year","quarter","remand_status","age_band","age_order"],
                          observed=True)["count"].sum().reset_index()
                 .sort_values(["quarter_end","remand_status","age_order"]))
save_gold(remand_age, "gold_remand_age")

def r_summary(df):
    valid = df["count"].sum()
    yr1   = df.loc[df.age_band.isin(["1 to 2 years","2 years or more"]), "count"].sum()
    return pd.Series({"open_valid": valid, "open_1yr_plus": yr1,
                      "share_1yr_plus": (yr1 / valid) if valid else float("nan")})

remand_sum = (rem.groupby(["quarter_end","year","quarter","remand_status"], observed=True)
                 .apply(r_summary, include_groups=False).reset_index()
                 .sort_values(["quarter_end","remand_status"]))
save_gold(remand_sum, "gold_remand_age_summary")
remand_sum[remand_sum.quarter_end == remand_sum.quarter_end.max()]

## Step 7 — `gold_waiting_hearing` (months waiting, hours in court)

**Question:** how long does a case wait for its Crown Court hearing, and how long is the
hearing itself?

**How I build it:** from `silver_cc_waiting_hearing`, national (`England and Wales`),
`All cases closed`, `All offences`, `All pleas`, `All remand statuses`, and the `Mean`/
`Median` stats. I pivot into one row per quarter with clearly-named, unit-tagged columns
(`waiting_mean_weeks`, `hearing_mean_hours`, …) so the six-months-vs-four-hours contrast
is impossible to misread.

In [ ]:
wh = pd.read_parquet(SILVER / "silver_cc_waiting_hearing.parquet",
        columns=["quarter_end","year","quarter","region","case_type","offence",
                 "plea","remand_status","metric","stat","unit","value"])
w = wh[(wh.region == "England and Wales") & (wh.case_type == "All cases closed") &
       (wh.offence == "All offences") & (wh.plea == "All pleas") &
       (wh.remand_status == "All remand statuses") & (wh.stat.isin(["Mean","Median"]))].copy()

# build a tidy, unit-tagged column name, e.g. "waiting_mean_weeks"
w["col"] = (w["metric"].str.replace(" times", "", regex=False).str.lower()
            + "_" + w["stat"].str.lower() + "_" + w["unit"])
wait_hear = (w.pivot_table(index=["quarter_end","year","quarter"], columns="col",
                           values="value", aggfunc="first").reset_index())
wait_hear.columns.name = None
wait_hear = wait_hear.sort_values("quarter_end").reset_index(drop=True)

save_gold(wait_hear, "gold_waiting_hearing")
wait_hear.tail(4)

## Step 8 — `gold_timeliness_by_region` (the geography of the wait)

**Question:** does the end-to-end wait depend on the region a case is heard in? From
`silver_cc_timeliness`, national + regional, `All cases closed`, `All offences`, keeping
the offence-to-completion median and mean (days), one row per quarter × region.

In [ ]:
tl = pd.read_parquet(SILVER / "silver_cc_timeliness.parquet",
        columns=["quarter_end","year","quarter","geo_area","geo_level","case_type","offence",
                 "offence_to_completion_median","offence_to_completion_mean"])
tr = tl[(tl.geo_level.isin(["region","national"])) & (tl.case_type == "All cases closed") &
        (tl.offence == "All offences")]
tr = (tr.rename(columns={"geo_area": "region"})
        [["quarter_end","year","quarter","region","geo_level",
          "offence_to_completion_median","offence_to_completion_mean"]]
        .sort_values(["quarter_end","offence_to_completion_median"]))
save_gold(tr, "gold_timeliness_by_region")
tr[tr.quarter_end == tr.quarter_end.max()]

## Step 9 — `gold_oldest_by_region` (how old the pile is, by region)

Share of open cases waiting a year or more, per region per quarter, from
`silver_cc_open_age` (national + regional, `All open cases`, `All offences`, real age
bands only — the `Total`/`Valid` summary rows excluded).

In [ ]:
oa = pd.read_parquet(SILVER / "silver_cc_open_age.parquet",
        columns=["quarter_end","year","quarter","geo_area","geo_level","case_type",
                 "offence","age_band","row_type","count"])
ob = oa[(oa.geo_level.isin(["region","national"])) & (oa.case_type == "All open cases") &
        (oa.offence == "All offences") & (oa.row_type == "age_band")]
def rshare(df):
    valid = df["count"].sum()
    yr1 = df.loc[df.age_band.isin(["1 to 2 years","2 years or more"]), "count"].sum()
    return pd.Series({"valid_cases": valid, "open_1yr_plus": yr1,
                      "share_1yr_plus": (yr1 / valid) if valid else float("nan")})
reg = (ob.groupby(["quarter_end","year","quarter","geo_area","geo_level"], observed=True)
         .apply(rshare, include_groups=False).reset_index().rename(columns={"geo_area": "region"})
         .sort_values(["quarter_end","share_1yr_plus"], ascending=[True, False]))
save_gold(reg, "gold_oldest_by_region")
reg[reg.quarter_end == reg.quarter_end.max()]

## Step 10 — `gold_court_backlog_ratio` (backlog pressure by court)

`silver_cc_rdos` is the only table with court-level detail. For each court I take its
latest open caseload and divide by the cases it clears in a year — "how long the pile
would take to clear at today's pace." Tiny courts (<300 disposals/yr) and `Unknown` are
dropped so the ratio isn't noise.

In [ ]:
rc = pd.read_parquet(SILVER / "silver_cc_rdos.parquet",
        columns=["year","quarter","region","court","offence","measure","count"])
rc = rc[rc.offence == "All offences"]
LY = rc.year.max()
op = rc[(rc.year == LY) & (rc.quarter == "Q4") & (rc.measure == "Open")].groupby(["region","court"], observed=True)["count"].sum()
di = rc[(rc.year == LY) & (rc.measure == "Disposals")].groupby(["region","court"], observed=True)["count"].sum()
court = pd.concat([op.rename("open_latest"), di.rename("annual_disposals")], axis=1).reset_index().dropna()
court = court[(court.court != "Unknown") & (court.annual_disposals >= 300)]
court["years_of_backlog"] = court["open_latest"] / court["annual_disposals"]
court = court.sort_values("years_of_backlog", ascending=False).reset_index(drop=True)
save_gold(court, "gold_court_backlog_ratio")
print(f"latest year: {LY} · {len(court)} courts · median {court.years_of_backlog.median():.2f} yrs")
court.head(6)

## Step 11 — Validate the Gold cuts against known facts

Three sanity checks tie Gold back to the source story:
1. `gold_backlog_trajectory` — open caseload in the latest quarter = **80,203**.
2. `gold_caseload_age_summary` — `open_1yr_plus` in the latest quarter = **21,002**.
3. `gold_oldest_by_offence` — sexual offences + violence against the person make up
   **over half** of the latest quarter's 1yr-plus cases (the headline claim).

In [ ]:
traj = pd.read_parquet(GOLD / "gold_backlog_trajectory.parquet")
summ = pd.read_parquet(GOLD / "gold_caseload_age_summary.parquet")
byof = pd.read_parquet(GOLD / "gold_oldest_by_offence.parquet")

L = traj.quarter_end.max()
open_latest = int(traj.loc[traj.quarter_end == L, "open"].iloc[0])
yr1_latest  = int(summ.loc[summ.quarter_end == L, "open_1yr_plus"].iloc[0])

lo = byof[byof.quarter_end == L]
sv = lo.loc[lo.offence.isin(["Sexual offences","Violence against the person"]),
            "open_1yr_plus"].sum()
share = sv / lo["open_1yr_plus"].sum()

print(f"latest quarter                         : {L.date()}")
print(f"open caseload (trajectory)             : {open_latest:,}   (expect 80,203)")
print(f"open 1yr+ (age summary)                : {yr1_latest:,}   (expect 21,002)")
print(f"sexual + violence share of 1yr+ cases  : {share:.1%}   (expect > 50%)")

# remand: bail cases age MORE than custody (the counter-intuitive finding)
rs = pd.read_parquet(GOLD / "gold_remand_age_summary.parquet")
rl = rs[rs.quarter_end == rs.quarter_end.max()].set_index("remand_status")
cust, bail = rl.loc["Custody", "share_1yr_plus"], rl.loc["Bail", "share_1yr_plus"]
print(f"1yr+ share — custody {cust:.1%} vs bail {bail:.1%}   (expect bail > custody)")

# waiting vs hearing: months waiting, hours in court
wht = pd.read_parquet(GOLD / "gold_waiting_hearing.parquet")
wl = wht[wht.quarter_end == wht.quarter_end.max()].iloc[0]
print(f"latest mean waiting {wl['waiting_mean_weeks']:.1f} weeks vs hearing {wl['hearing_mean_hours']:.1f} hours")

assert open_latest == 80203
assert yr1_latest == 21002
assert share > 0.50
assert bail > cust                       # bail caseload ages more than custody
assert wl["waiting_mean_weeks"] > 20 and wl["hearing_mean_hours"] < 10

# geography: South East slower / older than Wales
tbr = pd.read_parquet(GOLD / "gold_timeliness_by_region.parquet")
tl_ = tbr[tbr.quarter_end == tbr.quarter_end.max()].set_index("region")["offence_to_completion_median"]
obr = pd.read_parquet(GOLD / "gold_oldest_by_region.parquet")
ob_ = obr[obr.quarter_end == obr.quarter_end.max()].set_index("region")["share_1yr_plus"]
cb = pd.read_parquet(GOLD / "gold_court_backlog_ratio.parquet")
print(f"region timeliness — South East {tl_['South East']:.0f}d vs Wales {tl_['Wales']:.0f}d")
print(f"region 1yr+ share — South East {ob_['South East']:.1%} vs Wales {ob_['Wales']:.1%}")
print(f"court ratio — worst {cb.iloc[0]['court']} {cb.iloc[0]['years_of_backlog']:.2f}y vs best {cb.iloc[-1]['court']} {cb.iloc[-1]['years_of_backlog']:.2f}y")
assert tl_["South East"] > tl_["Wales"] and ob_["South East"] > ob_["Wales"]
print("\nAll Gold checks passed — the story cuts reconcile with the source. ")

---
### Gold layer complete

`data/gold/` now holds the story-ready cuts (Parquet + CSV): the national backlog
trajectory, the ageing caseload and its summary, the oldest cases by offence, the
end-to-end timeliness trend, the court-level geography, the custody-vs-bail remand
ageing, and the waiting-vs-hearing contrast. Each reconciles with the published figures.

**From here:** these five tables are the backbone of the article series and a Streamlit
app — one solid pipeline, several human stories. Next I can wire up the Streamlit app on
top of `data/gold/`, or start drafting the first article from `gold_backlog_trajectory`
and `gold_caseload_age`.